In [35]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch08_Attention(스마트번역기)</font>**
- Google Neural Machine Translation(GNMT)
- RNN기반의 Seq2Seq방식
- 인코더입력/디코더입력(모범답안) - 디코더 출력(답안) ; 인코더와 디코더가 연결된 구조

# 1. 패키지 import 및 하이퍼 파라미터


In [36]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [37]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor=raw.values.tolist()
print(eng_kor[:3])
print('학습할 영-한 데이터 갯수:', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
학습할 영-한 데이터 갯수: 110


In [38]:
e_alpha = [ c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([ data[1] for data in eng_kor])
k_ch= list(set([ch for ch in korean]))
k_ch.sort()
k_alpha= k_ch

In [39]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 : ',alpha)
alpha_total_size= len(alpha)
print('전체 알파벳 갯수(원핫인코딩 사이즈):', alpha_total_size)

영어와 한글 알파벳 :  ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳 갯수(원핫인코딩 사이즈): 171


# 3. 문자당 num을 갖는 dict
- 전예제 : {'the':1, 'a':2,...} / {1:'the',2:'a',...}
- {'S':0, 'E':1,...}

In [40]:
# char_to_num={}
# for i, ch in enumerate(alpha):
#     char_to_num[ch] = i
char_to_num={ch: i for i, ch in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [41]:
# 문자-> 숫자 / 숫자-> 문자
print('문자->숫자:', char_to_num.get('c'))
print('숫자->문자:', alpha[5])

문자->숫자: 5
숫자->문자: c


In [42]:
data = eng_kor[0]
print(data)
print('인코더 입력(원핫인코딩전):', [char_to_num.get(ch) for ch in data[0]])
print('디코더 입력(원핫인코딩전):', [char_to_num.get(ch) for ch in 'S' + data[1]] )
print('디코더 출력:',[char_to_num.get(ch) for ch in data[1]+ 'E'])

['cold', '감기']
인코더 입력(원핫인코딩전): [5, 17, 14, 6]
디코더 입력(원핫인코딩전): [0, 32, 46]
디코더 출력: [32, 46, 1]


In [ ]:
# 원핫인코딩 방법1
pd.get_dummies([5, 17, 14, 6])

In [ ]:
# 원핫인코딩 방법2
to_categorical([5, 17, 14, 6], num_classes=alpha_total_size)

In [43]:
#원핫인코딩 방법3 : np.eye(n) - n행 n열 단위행렬
np.eye(alpha_total_size)[[5, 17, 14, 6]]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.,

# 4. 인코더입력, 디코더입력, 디코더출력
- 인코더입력과 디코더입력(원핫인코딩), 디코더출력(원핫인코딩X-loss를 sparse categoricalcrossentropy)


In [44]:
def encoding(eng_kor=eng_kor):
    '인코더입력데이터(110*4*171), 디코더입력데이터(110*3*171), 디코더출력데이터를 return'  # 3개 축이 다 동일해야함
       # 영어단어 110 , 문자4개, 원핫인코더 171
    enc_in=[] #인코더입력(cold 원핫인코딩)
    dec_in=[] #디코더입력(S 감기 원핫인코딩)
    dec_out=[] #디코더입력(감기E 라벨인코딩)
    for data in eng_kor:
        #인코더 입력(영어->숫자->원핫인코딩)
        eng=[char_to_num.get(ch) for ch in data[0]]
        eng_one=to_categorical(eng, num_classes=alpha_total_size)
        enc_in.append(eng_one)
        #디코더 입력('S'한글 -> 숫자 -> 원핫인코딩)
        kor =[char_to_num.get(ch) for ch in 'S'+ data[1]]
        kor_one=np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        #디코더 출력(한글'E' -> 숫자)
        kor = [[char_to_num.get(ch)] for ch in data[1]+'E']
        dec_out.append(kor)
        #print(kor)
     #인공신경망에 넣을 데이터이므로 numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    #print(enc_in.shape, dec_in.shape, dec_out.shape)
    return enc_in, dec_in, dec_out

sample=[['cold', '감기'],['come', '오다']]
X_enc,X_dec,Y_dec=encoding(sample)
X_enc.shape,X_dec.shape, Y_dec.shape
    

((2, 4, 171), (2, 3, 171), (2, 3, 1))

# 5. 전체 번역데이터(독립변수,타겟변수)

In [51]:
#Seq2Seq에 들어갈 데이터
X_enc, X_dec, Y_dec = encoding(eng_kor)

In [52]:
X_enc.shape,X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

# 6. 모델구현(Attention) 추가


In [53]:
from tensorflow.keras.layers import Attention, Concatenate
#인코더 LSTM 구현
ENC_IN = Input(shape=(4, alpha_total_size))
ENC_OUT, state_h, state_c =LSTM(units=MY_HIDDEN,
                                return_sequences=True, #위 출력
                                return_state=True)(ENC_IN) #h와 c옆 출력
#디코더 LSTM구현
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID, _, _= LSTM(units=MY_HIDDEN,
                    return_sequences=True,
                    return_state=True)(DEC_IN,
                                      initial_state=[state_h, state_c])
#어텐션 메커니즘
CONTEXT_VECTOR = Attention()([DEC_MID, ENC_OUT])
#CONTEXT_VECTOR와 디코더 LSTM 출력을 결합
CONTEXT_AND_LSTM = Concatenate()([CONTEXT_VECTOR, DEC_MID])
#최종 출력층
OUT=Dense(units=alpha_total_size, activation='softmax')(CONTEXT_AND_LSTM)
#모델
model=Model(inputs = [ENC_IN, DEC_IN],
           outputs=OUT,
           name='model')
model.summary

<bound method Model.summary of <keras.engine.functional.Functional object at 0x000002604A54FCA0>>

# 7. 모델 학습


In [54]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='rmsprop',
             metrics=['accuracy'])
begin= time()
hist = model.fit([X_enc,X_dec],Y_dec,
                epochs=MY_EPOCH,
                verbose=2)
end= time()
print(f'학습시간 : {end-begin:.2f}초')

Epoch 1/500
4/4 - 4s - loss: 5.1108 - accuracy: 0.2212 - 4s/epoch - 1s/step
Epoch 2/500
4/4 - 0s - loss: 4.9587 - accuracy: 0.3333 - 33ms/epoch - 8ms/step
Epoch 3/500
4/4 - 0s - loss: 4.2977 - accuracy: 0.3333 - 36ms/epoch - 9ms/step
Epoch 4/500
4/4 - 0s - loss: 3.4672 - accuracy: 0.3333 - 34ms/epoch - 9ms/step
Epoch 5/500
4/4 - 0s - loss: 3.4013 - accuracy: 0.3333 - 35ms/epoch - 9ms/step
Epoch 6/500
4/4 - 0s - loss: 3.3567 - accuracy: 0.3333 - 35ms/epoch - 9ms/step
Epoch 7/500
4/4 - 0s - loss: 3.3226 - accuracy: 0.3333 - 36ms/epoch - 9ms/step
Epoch 8/500
4/4 - 0s - loss: 3.2882 - accuracy: 0.3333 - 35ms/epoch - 9ms/step
Epoch 9/500
4/4 - 0s - loss: 3.2614 - accuracy: 0.3333 - 35ms/epoch - 9ms/step
Epoch 10/500
4/4 - 0s - loss: 3.2297 - accuracy: 0.3394 - 33ms/epoch - 8ms/step
Epoch 11/500
4/4 - 0s - loss: 3.2025 - accuracy: 0.3394 - 35ms/epoch - 9ms/step
Epoch 12/500
4/4 - 0s - loss: 3.1840 - accuracy: 0.3394 - 34ms/epoch - 9ms/step
Epoch 13/500
4/4 - 0s - loss: 3.1699 - accuracy: 0.3

Epoch 104/500
4/4 - 0s - loss: 0.1797 - accuracy: 0.9939 - 34ms/epoch - 8ms/step
Epoch 105/500
4/4 - 0s - loss: 0.1738 - accuracy: 0.9909 - 40ms/epoch - 10ms/step
Epoch 106/500
4/4 - 0s - loss: 0.1731 - accuracy: 0.9909 - 37ms/epoch - 9ms/step
Epoch 107/500
4/4 - 0s - loss: 0.1602 - accuracy: 0.9970 - 36ms/epoch - 9ms/step
Epoch 108/500
4/4 - 0s - loss: 0.1507 - accuracy: 0.9909 - 36ms/epoch - 9ms/step
Epoch 109/500
4/4 - 0s - loss: 0.1459 - accuracy: 0.9879 - 36ms/epoch - 9ms/step
Epoch 110/500
4/4 - 0s - loss: 0.1320 - accuracy: 0.9939 - 32ms/epoch - 8ms/step
Epoch 111/500
4/4 - 0s - loss: 0.1254 - accuracy: 0.9970 - 35ms/epoch - 9ms/step
Epoch 112/500
4/4 - 0s - loss: 0.1232 - accuracy: 0.9970 - 36ms/epoch - 9ms/step
Epoch 113/500
4/4 - 0s - loss: 0.1194 - accuracy: 0.9939 - 38ms/epoch - 10ms/step
Epoch 114/500
4/4 - 0s - loss: 0.1195 - accuracy: 0.9879 - 36ms/epoch - 9ms/step
Epoch 115/500
4/4 - 0s - loss: 0.1028 - accuracy: 0.9939 - 34ms/epoch - 9ms/step
Epoch 116/500
4/4 - 0s - l

Epoch 205/500
4/4 - 0s - loss: 0.0012 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 206/500
4/4 - 0s - loss: 6.3194e-04 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 207/500
4/4 - 0s - loss: 6.0868e-04 - accuracy: 1.0000 - 38ms/epoch - 10ms/step
Epoch 208/500
4/4 - 0s - loss: 5.8943e-04 - accuracy: 1.0000 - 34ms/epoch - 9ms/step
Epoch 209/500
4/4 - 0s - loss: 5.5748e-04 - accuracy: 1.0000 - 42ms/epoch - 11ms/step
Epoch 210/500
4/4 - 0s - loss: 6.0924e-04 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 211/500
4/4 - 0s - loss: 4.9610e-04 - accuracy: 1.0000 - 38ms/epoch - 9ms/step
Epoch 212/500
4/4 - 0s - loss: 4.7910e-04 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 213/500
4/4 - 0s - loss: 4.4260e-04 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 214/500
4/4 - 0s - loss: 4.2509e-04 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 215/500
4/4 - 0s - loss: 6.1724e-04 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 216/500
4/4 - 0s - loss: 0.0121 - accuracy: 0.9939 - 35ms/epo

Epoch 302/500
4/4 - 0s - loss: 4.7676e-06 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 303/500
4/4 - 0s - loss: 4.6878e-06 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 304/500
4/4 - 0s - loss: 4.5794e-06 - accuracy: 1.0000 - 40ms/epoch - 10ms/step
Epoch 305/500
4/4 - 0s - loss: 4.3121e-06 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 306/500
4/4 - 0s - loss: 4.1481e-06 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 307/500
4/4 - 0s - loss: 4.1160e-06 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 308/500
4/4 - 0s - loss: 4.0022e-06 - accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 309/500
4/4 - 0s - loss: 3.8215e-06 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 310/500
4/4 - 0s - loss: 3.6644e-06 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 311/500
4/4 - 0s - loss: 3.6128e-06 - accuracy: 1.0000 - 42ms/epoch - 10ms/step
Epoch 312/500
4/4 - 0s - loss: 3.4596e-06 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 313/500
4/4 - 0s - loss: 3.3133e-06 - accuracy: 1.0000 -

4/4 - 0s - loss: 7.6872e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 399/500
4/4 - 0s - loss: 7.7341e-07 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 400/500
4/4 - 0s - loss: 7.6258e-07 - accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 401/500
4/4 - 0s - loss: 7.5427e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 402/500
4/4 - 0s - loss: 7.4343e-07 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 403/500
4/4 - 0s - loss: 7.2645e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 404/500
4/4 - 0s - loss: 7.3585e-07 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 405/500
4/4 - 0s - loss: 7.2392e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 406/500
4/4 - 0s - loss: 7.1706e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 407/500
4/4 - 0s - loss: 7.0875e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 408/500
4/4 - 0s - loss: 7.0153e-07 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 409/500
4/4 - 0s - loss: 6.9322e-07 - accuracy: 1.0000 - 36ms/epoch - 9m

Epoch 495/500
4/4 - 0s - loss: 3.7424e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 496/500
4/4 - 0s - loss: 3.6955e-07 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 497/500
4/4 - 0s - loss: 3.6846e-07 - accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 498/500
4/4 - 0s - loss: 3.6594e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 499/500
4/4 - 0s - loss: 3.6738e-07 - accuracy: 1.0000 - 37ms/epoch - 9ms/step
Epoch 500/500
4/4 - 0s - loss: 3.6449e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
학습시간 : 23.83초


In [55]:
model.evaluate([X_enc,X_dec],Y_dec)

4/4 [==============================] - 1s 5ms/step - loss: 3.6124e-07 - accuracy: 1.0000


[3.612401542341104e-07, 1.0]

# 8. 모델 사용

In [56]:
# 쉬운문제
easy_test=[['very','pp'],
           ['wave','pp'],
           ['wood','pp'],
           ['sign','pp'],
           ['thin','pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [57]:
pred = model.predict([enc_in,dec_in])
y_hat=pred.argmax(axis=-1)

1/1 [==============================] - 1s 803ms/step


In [58]:
for i in range(len(easy_test)):
    eng = easy_test[i][0]
    hat=pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

very => 매우[ 75 124]
wave => 파도[157  62]
wood => 나무[48 83]
sign => 싸인[110 135]
thin => 얇은[113 129]


In [59]:
# 어려운 문제
easy_test=[['vrey','pp'],
           ['waev','pp'],
           ['wodd','pp'],
           ['sigg','pp'],
           ['tine','pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [60]:
pred = model.predict([enc_in,dec_in])
y_hat=pred.argmax(axis=-1)

1/1 [==============================] - 0s 27ms/step


In [61]:
for i in range(len(easy_test)):
    eng = easy_test[i][0]
    hat=pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

vrey => 매우[ 75 124]
waev => 파도[157  62]
wodd => 단어[ 61 114]
sigg => 싸인[110 135]
tine => 작은[139 129]
